In [1]:
%tensorflow_version 2.x

UsageError: Line magic function `%tensorflow_version` not found.


In [2]:
from __future__ import absolute_import, division, print_function, unicode_literals

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from six.moves import urllib

import tensorflow.compat.v2.feature_column as fc
import tensorflow as tf

In [4]:
dftrain = pd.read_csv('train.csv')
dftest = pd.read_csv('eval.csv')
data_train = dftrain.dropna()
data_test = dftest.dropna()
y_train = dftrain.pop('survived')
y_test = dftest.pop('survived')
data_train.drop(['sex','class','deck','alone','embark_town'],axis=1,inplace=True,)
data_test.drop(['sex','class','deck','alone','embark_town'],axis=1,inplace=True,)
y_log_train = data_train.pop('survived')
y_log_test = data_test.pop('survived')
x_log_train = data_train
x_log_test = data_test

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [7]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 627 entries, 0 to 626
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   age                 627 non-null    float64
 1   n_siblings_spouses  627 non-null    int64  
 2   parch               627 non-null    int64  
 3   fare                627 non-null    float64
dtypes: float64(2), int64(2)
memory usage: 19.7 KB


In [8]:
LM=LogisticRegression()
LM.fit(x_log_train,y_log_train)
LinM = LinearRegression()
LinM.fit(x_log_train, y_log_train)

LinearRegression()

In [9]:
predictions_log = LM.predict(x_log_test)
predictions_lin = LinM.predict(x_log_test)

In [10]:
from sklearn.metrics import classification_report
print(classification_report(y_log_test,predictions_log))

              precision    recall  f1-score   support

           0       0.68      0.96      0.80       165
           1       0.78      0.25      0.38        99

    accuracy                           0.69       264
   macro avg       0.73      0.61      0.59       264
weighted avg       0.72      0.69      0.64       264



In [11]:
dftrain.head()

,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,male,22.0,1,0,7.2500,Third,unknown,Southampton,n
1,female,38.0,1,0,71.2833,First,C,Cherbourg,n
2,female,26.0,0,0,7.9250,Third,unknown,Southampton,y
3,female,35.0,1,0,53.1000,First,C,Southampton,n
4,male,28.0,0,0,8.4583,Third,unknown,Queenstown,y


In [12]:
NUMERIC_COLS = ['age', 'fare']
CATEGORICAL_COLS = ['sex', 'parch', 'alone', 'embark_town', 'deck', 'class', 'n_siblings_spouses']

In [13]:
dftrain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 627 entries, 0 to 626
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   sex                 627 non-null    object 
 1   age                 627 non-null    float64
 2   n_siblings_spouses  627 non-null    int64  
 3   parch               627 non-null    int64  
 4   fare                627 non-null    float64
 5   class               627 non-null    object 
 6   deck                627 non-null    object 
 7   embark_town         627 non-null    object 
 8   alone               627 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 44.2+ KB


In [15]:
feature_cols = []
for feature_name in CATEGORICAL_COLS:
    vocab = dftrain[feature_name].unique()  # gets a list of all unique values from given feature column
    feature_cols.append(tf.feature_column.categorical_column_with_vocabulary_list(feature_name, vocab))

for feature_name in NUMERIC_COLS:
    feature_cols.append(tf.feature_column.numeric_column(feature_name, dtype=tf.float32))



In [20]:
def make_input_fn(data_df, label_df, num_epochs=30, shuffle=True, batch_size=32):
    def input_function():  # inner function, this will be returned
        ds = tf.data.Dataset.from_tensor_slices((dict(data_df), label_df))  # create tf.data.Dataset object with data and its label
        if shuffle:
            ds = ds.shuffle(1000)  # randomize order of data
        ds = ds.batch(batch_size).repeat(num_epochs)  # split dataset into batches of 32 and repeat process for number of epochs
        return ds  # return a batch of the dataset
    return input_function  
    # return a function object for use

train_input_fn = make_input_fn(dftrain, y_train)  # here we will call the input_function that was returned to us to get a dataset object we can feed to the model
test_input_fn = make_input_fn(dftest, y_test, num_epochs=1, shuffle=False)


In [21]:
linear_est = tf.estimator.LinearClassifier(feature_columns=feature_cols)
# We create a linear estimtor by passing the feature columns we created earlier
linear_est.train(train_input_fn)


INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': '/var/folders/pv/mgx83zhn56z6qhm7srylph6h0000gn/T/tmp443i3np9', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
INFO:tensorflow:Calling model_fn.
INF

In [22]:
result = linear_est.evaluate(test_input_fn)
print(result['accuracy'])

INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Starting evaluation at 2022-03-08T19:52:27
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from /var/folders/pv/mgx83zhn56z6qhm7srylph6h0000gn/T/tmp443i3np9/model.ckpt-600
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Inference Time : 1.25596s
INFO:tensorflow:Finished evaluation at 2022-03-08-19:52:28
INFO:tensorflow:Saving dict for global step 600: accuracy = 0.7765151, accuracy_baseline = 0.625, auc = 0.8373431, auc_precision_recall = 0.77947533, average_loss = 0.47046706, global_step = 600, label/mean = 0.375, loss = 0.45806038, precision = 0.7, prediction/mean = 0.3798957, recall = 0.7070707
INFO:tensorflow:Saving 'checkpoint_path' summary for global step 600: /var/folders/pv/mgx83zhn56z6qhm7srylph6h0000gn/T/tmp443i3np9/model.ckpt-600
0.7765151


In [23]:
result = list(linear_est.predict(test_input_fn))

INFO:tensorflow:Calling model_fn.


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/tensorflow_estimator/python/estimator/canned/linear.py:1468: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use `layer.add_weight` method instead.
  self.bias = self.add_variable(


INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from /var/folders/pv/mgx83zhn56z6qhm7srylph6h0000gn/T/tmp443i3np9/model.ckpt-600
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.


In [36]:
for n in range(10):
    #print(dftest.loc[n])
    print(y_test.loc[n])
    print(result[n]['probabilities'][1])

0
0.052729085
0
0.39671007
1
0.65584147
1
0.52501184
1
0.2225721
1
0.7588493
0
0.48764956
0
0.091856234
0
0.6374843
1
0.72597563


In [37]:
CSV_COLUMN_NAMES = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Species']
SPECIES = ['Setosa', 'Versicolor', 'Virginica']
# Lets define some constants to help us later on

In [38]:
train_path = tf.keras.utils.get_file(
    "iris_training.csv", "https://storage.googleapis.com/download.tensorflow.org/data/iris_training.csv")
test_path = tf.keras.utils.get_file(
    "iris_test.csv", "https://storage.googleapis.com/download.tensorflow.org/data/iris_test.csv")

train = pd.read_csv(train_path, names=CSV_COLUMN_NAMES, header=0)
test = pd.read_csv(test_path, names=CSV_COLUMN_NAMES, header=0)
# Here we use keras (a module inside of TensorFlow) to grab our datasets and read them into a pandas dataframe

Exception: URL fetch failure on https://storage.googleapis.com/download.tensorflow.org/data/iris_training.csv: None -- [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1108)